# 06 - Business Analysis


In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.max_columns', 50)

from src.data.load_data import load_csv
from src.analysis.kpi_analysis import calculate_all_kpis
from src.analysis.business_analysis import (
    defect_rate_by_bucket, cost_comparison_defective_vs_nondefective,
    downtime_vs_production_performance, maintenance_vs_quality,
    supplier_quality_vs_performance, energy_efficiency_overview,
    worker_productivity_vs_defects, inventory_and_stockout_overview,
)
from src.utils.config import load_config, resolve_path

config = load_config()
df = load_csv(resolve_path(config['data']['processed_path']))
print(f"Working with processed dataset: {df.shape}")

2026-09-20 22:34:55 | INFO     | src.data.load_data | Loaded CSV 'manufacturing_processed.csv' with shape (3240, 21)


Working with processed dataset: (3240, 21)


## Manufacturing KPI Summary



In [2]:
kpis = calculate_all_kpis(df)
kpi_table = pd.Series(kpis).rename('Value').to_frame()
kpi_table

2026-09-20 22:37:08 | INFO     | src.analysis.kpi_analysis | Calculated 17 manufacturing KPIs.


,Value
Overall Defect Rate (%),8.404000e+01
Average DefectRate Metric,2.749000e+00
Total Production Volume,1.777215e+06
Average Production Volume,5.485200e+02
Total Production Cost ($),4.025058e+07
Average Production Cost ($),1.242302e+04
Average Cost per Unit ($),3.206000e+01
Average Quality Score,8.013000e+01
Average Downtime (%),2.501000e+00
Average Maintenance Hours,1.148000e+01


**Interpretation:** The overall defect rate of 84.0% is the headline quality metric. Average production cost is roughly $12,423 per record, and average quality score is roughly 79.9 out of 100. These KPIs form the baseline against which any future improvement should be measured.

## Q1: What is the overall defect rate?

**Answer:** already computed above - **84.0%** of the 3,240 production records are flagged as defective. This is the single most important quality KPI in this dataset.

## Q2: Which factors are associated with product defects?

I use bucketed defect-rate analysis across the two variables identified in the EDA notebook as most strongly correlated with the defect outcome: `MaintenanceHours` and `QualityScore`.

In [3]:
defect_rate_by_bucket(df, 'MaintenanceHours')

,MaintenanceHours_range,record_count,defect_rate_pct
0,"(-0.023, 4.6]",678,70.50
1,"(4.6, 9.2]",653,70.90
2,"(9.2, 13.8]",557,86.54
3,"(13.8, 18.4]",696,95.55
4,"(18.4, 23.0]",656,96.80


In [4]:
defect_rate_by_bucket(df, 'QualityScore')

,QualityScore_range,record_count,defect_rate_pct
0,"(59.97, 68.007]",649,95.69
1,"(68.007, 76.005]",627,93.46
2,"(76.005, 84.002]",660,76.36
3,"(84.002, 92.0]",636,77.99
4,"(92.0, 99.997]",668,77.25


**Interpretation:** The defect rate rises sharply as `MaintenanceHours` increases - from roughly 70% in the lowest maintenance-hours bucket to roughly 97% in the highest bucket. Conversely, the defect rate falls as `QualityScore` increases. These two variables are the clearest, most actionable factors associated with defects in this dataset.

## Q3: Are defects associated with increased operational costs?

In [5]:
cost_comparison_defective_vs_nondefective(df)

,avg_production_cost,avg_cost_per_unit,avg_production_volume,record_count
DefectStatus,,,,
Non-Defective,12158.88,36.28,470.87,517
Defective,12473.17,31.26,563.27,2723


**Interpretation:** Average production cost and average cost per unit are similar between defective and non-defective records in this dataset - cost does not appear to be a strong differentiator of defect status here, unlike maintenance hours and quality score.

## Q4: How does equipment downtime affect production performance?

In [6]:
downtime_vs_production_performance(df)

,DowntimePercentage_range,avg_production_volume,defect_rate_pct,record_count
0,"(-0.00333, 1.001]",538.37,84.04,639
1,"(1.001, 2.0]",542.88,83.21,685
2,"(2.0, 2.999]",549.38,84.85,614
3,"(2.999, 3.998]",565.15,84.32,644
4,"(3.998, 4.998]",547.19,83.89,658


**Interpretation:** Average production volume is broadly similar across downtime bands, but the defect rate shows some variation across downtime ranges, suggesting downtime has a more modest / less consistent relationship with defects than maintenance hours does in this dataset.

## Q5: Are maintenance-related variables associated with downtime or quality problems?

In [7]:
maintenance_vs_quality(df)

,MaintenanceHours_range,avg_quality_score,defect_rate_pct,record_count
0,"(-0.023, 4.6]",80.38,70.50,678
1,"(4.6, 9.2]",80.20,70.90,653
2,"(9.2, 13.8]",80.20,86.54,557
3,"(13.8, 18.4]",79.99,95.55,696
4,"(18.4, 23.0]",79.91,96.80,656


**Interpretation:** As `MaintenanceHours` increases, average `QualityScore` tends to decrease while defect rate increases - reinforcing the finding from Q2 that maintenance hours is one of the strongest signals in this dataset.

## Q6: How does supplier quality relate to manufacturing quality?

In [8]:
supplier_quality_vs_performance(df)

,SupplierQuality_range,avg_quality_score,defect_rate_pct,record_count
0,"(79.985, 84.002]",80.28,82.63,668
1,"(84.002, 87.999]",80.64,82.03,651
2,"(87.999, 91.995]",80.23,83.87,682
3,"(91.995, 95.992]",79.95,86.71,602
4,"(95.992, 99.989]",79.53,85.24,637


**Interpretation:** Average quality score and defect rate do not show a strong, consistent trend across `SupplierQuality` bands in this dataset - supplier quality score alone is a weaker predictor of outcomes here than maintenance hours or quality score.

## Q7: How efficient is energy consumption - does it differ for defective batches?

In [9]:
energy_efficiency_overview(df)

,avg_energy_consumption,avg_energy_efficiency,record_count
DefectStatus,,,
Non-Defective,2975.158,0.309,517
Defective,2991.027,0.298,2723


**Interpretation:** Average energy consumption and efficiency are broadly similar between defective and non-defective records - energy usage patterns do not show a strong association with the defect outcome in this dataset.

## Q8: Does worker productivity relate to defect rate?

In [10]:
worker_productivity_vs_defects(df)

,WorkerProductivity_range,defect_rate_pct,record_count
0,"(79.985, 84.003]",83.96,636
1,"(84.003, 88.002]",86.09,640
2,"(88.002, 92.0]",83.33,654
3,"(92.0, 95.998]",82.06,669
4,"(95.998, 99.997]",84.87,641


**Interpretation:** Defect rate does not show a strong, consistent trend across `WorkerProductivity` buckets in this dataset.

## Q9: Inventory turnover and stockout rate relationship

In [11]:
inventory_and_stockout_overview(df)

,avg_inventory_turnover,avg_stockout_rate,record_count
DefectStatus,,,
Non-Defective,5.9837,0.0482,517
Defective,6.0265,0.0514,2723


**Interpretation:** Average inventory turnover and stockout rate are broadly similar between defective and non-defective records - these inventory variables show weak association with the defect outcome in this dataset, consistent with them capturing a different operational dimension (supply continuity) rather than production quality.

## Summary of business analysis findings

1. **Overall defect rate: 84.0%** - the primary quality KPI for this dataset.
2. **`MaintenanceHours` is the strongest identified driver of defects** - defect rate rises from ~70% to ~97% as maintenance hours increase.
3. **`QualityScore` is strongly, inversely related to defects** - as expected, lower quality scores associate with more defective records.
4. Cost, downtime, supplier quality, worker productivity, energy, and inventory variables show comparatively weak or inconsistent relationships with the defect outcome in this specific dataset.
